# GRAST — Easy recovery trên Kaggle
Một vòng **1.000 successful updates** từ selected2200 trước WP09, cùng kit/recipe/tokenizer/partition với Colab. Luôn in log; dùng **GPU số 0**, không chạy distributed dù phiên có nhiều GPU.

1. Import notebook này vào Kaggle, chọn GPU accelerator.
2. Tạo/attach Dataset bằng **Add Input**, chứa `grast-easy-recovery-kit-20261127.zip` và checkpoint nguồn `selected_model.pt`. Kit ZIP có thể được Kaggle tự giải nén; notebook hỗ trợ cả hai dạng. Không cần Internet nếu runtime đã có PyTorch, NumPy, Matplotlib.
3. Chạy `SMOKE_ONLY=True` trước (2 updates, không chấm gate). Sau khi qua, đổi `False` rồi chạy lại các cell để train đủ 1.000 updates. Giữ recipe đã khóa, không chỉnh theo điểm gate.
4. Kết quả ở `/kaggle/working/grast-easy-recovery/`. **Lưu một notebook version có Output trước khi kết thúc phiên**; `/kaggle/working` không phải Google Drive. Tải summary ZIP và selected model từ Output. Để resume, attach Output đã lưu bằng Add Input và chỉ rõ đường dẫn `latest.pt`.

Validation selection16 và gate48 không trùng cặp toán hạng với nhau hoặc các phép toán con đã biết trong train/selection/pilot cũ. Đây vẫn là development validation từng dùng đo loss, không phải test chưa dùng. Không mở test set; chưa triển khai confidence policy.

In [ ]:
import hashlib
import importlib.util
import json
import os
import signal
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

os.environ.update(CUBLAS_WORKSPACE_CONFIG=":4096:8", PYTHONUNBUFFERED="1")
SOURCE_COMMIT = "1ff2079f5833345b10ad5f852e3a297f17d072ab"
TOTAL_UPDATES = 1000
KIT_NAME = "grast-easy-recovery-kit-20261127.zip"
KIT_SHA256 = "c8b15e13ab936af43a7efc649fedc1191f210fa4d9a097c95bc63bb68ab606cf"
MANIFEST_SHA256 = "d58c11c093e80035a627dad338956b70514a8ef9dfa8121a54ec3c0e3231af98"
SOURCE_SHA256 = "d2ea45abc8f81b516552c184d75c595b95fa2bd1abdc260538e497da4b627dc8"
SOURCE_BYTES = 201411596
SMOKE_ONLY = True
RESUME = ""  # Full easy-recovery latest.pt; not selected_model.pt or old curriculum latest.pt.
KIT_PATH = ""  # Optional explicit ZIP path or expanded directory under /kaggle/input.
SOURCE_CHECKPOINT = ""  # Optional explicit selected2200 file; otherwise discover by SHA256.
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/grast-easy-recovery")
OUTPUT_ROOT = WORK_ROOT / ("smoke_runs" if SMOKE_ONLY else "runs")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CONSOLE_LOG = WORK_ROOT / "console.log"
SCRATCH_ROOT = Path(tempfile.mkdtemp(prefix="grast-easy-kaggle-"))
SITE = SCRATCH_ROOT / "site"
CHILD_ENV = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1"}
LAST_RUN_DIRECTORY = None
EXPECTED_FILES = {
    "data/manifest.json",
    "data/train.jsonl",
    "data/selection.jsonl",
    "data/gate.jsonl",
    "tokenizer/vocab.json",
    "tokenizer/merges.json",
    "tokenizer/tokenizer_manifest.json",
    "tokenizer/tokenizer_config.json",
    "easy_recovery.toml",
    "grast-0.1.0-py3-none-any.whl",
    "bundle_manifest.json",
}


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def run_logged(command, cwd=None):
    global LAST_RUN_DIRECTORY
    text = "RUN: " + " ".join(map(str, command))
    print(text, flush=True)
    with CONSOLE_LOG.open("a", encoding="utf-8") as log:
        log.write(text + "\n")
        log.flush()
        with subprocess.Popen(
            list(map(str, command)),
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            bufsize=1,
            env=CHILD_ENV,
        ) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                    try:
                        record = json.loads(line)
                    except (ValueError, TypeError):
                        continue
                    if isinstance(record, dict) and record.get("event") in (
                        "easy_recovery_started",
                        "easy_recovery_finished",
                    ):
                        LAST_RUN_DIRECTORY = record["run_directory"]
                status = process.wait()
            except BaseException:
                if process.poll() is None:
                    process.send_signal(signal.SIGINT)
                    try:
                        process.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()
                raise
        if status:
            print("Last run directory:", LAST_RUN_DIRECTORY, flush=True)
            raise subprocess.CalledProcessError(status, command)
    return LAST_RUN_DIRECTORY

In [ ]:
def find_kit(input_root, explicit_path=""):
    if explicit_path:
        p = Path(explicit_path)
        if not p.exists():
            raise FileNotFoundError("Explicit kit path does not exist: " + str(p))
        candidates = [("expanded" if p.is_dir() else "zip", p)]
    else:
        root = Path(input_root)
        candidates = [("zip", p) for p in sorted(root.rglob(KIT_NAME)) if p.is_file()]
        for p in sorted(root.rglob("bundle_manifest.json")):
            if p.stat().st_size > 65536:
                continue
            try:
                m = json.loads(p.read_text(encoding="utf-8"))
            except (ValueError, UnicodeError):
                continue
            if isinstance(m, dict) and m.get("scope") == "easy-recovery-colab-v1":
                candidates.append(("expanded", p.parent))
    if not candidates:
        raise FileNotFoundError("Add Input the easy-recovery kit ZIP or expanded kit directory.")
    if len(candidates) != 1:
        raise ValueError("Multiple kits found; set KIT_PATH explicitly.")
    kind, p = candidates[0]
    checked = p if kind == "zip" else p / "bundle_manifest.json"
    if not checked.is_file() or file_sha256(checked) != (
        KIT_SHA256 if kind == "zip" else MANIFEST_SHA256
    ):
        raise ValueError("Kit SHA256 mismatch: " + str(checked))
    return kind, p


def prepare_kit(input_root, scratch_root, explicit_path=""):
    kind, root = find_kit(input_root, explicit_path)
    if kind == "zip":
        with zipfile.ZipFile(root) as archive:
            infos, names = archive.infolist(), archive.namelist()
            if (
                set(names) != EXPECTED_FILES
                or len(set(names)) != len(names)
                or sum(i.file_size for i in infos) > 256 * 1024**2
                or archive.getinfo("bundle_manifest.json").file_size > 65536
                or any((i.external_attr >> 16) & 0o170000 == 0o120000 for i in infos)
            ):
                raise ValueError("Unsafe/duplicate/oversized kit entries")
            root = Path(scratch_root) / "kit"
            root.mkdir(parents=True, exist_ok=False)
            archive.extractall(root)
    if file_sha256(root / "bundle_manifest.json") != MANIFEST_SHA256:
        raise ValueError("Kit manifest SHA256 mismatch")
    manifest = json.loads((root / "bundle_manifest.json").read_text(encoding="utf-8"))
    if (
        manifest.get("scope") != "easy-recovery-colab-v1"
        or set(manifest.get("files", {})) != EXPECTED_FILES - {"bundle_manifest.json"}
        or manifest.get("source_commit") != SOURCE_COMMIT
    ):
        raise ValueError("Kit manifest identity mismatch")
    for name, digest in manifest["files"].items():
        p = root / name
        if (
            p.is_symlink()
            or not p.resolve().is_relative_to(root.resolve())
            or not p.is_file()
            or file_sha256(p) != digest
        ):
            raise ValueError("Kit payload SHA256 mismatch: " + name)
    return root


def resolve_source(input_root, expected_sha256, explicit_path=""):
    if explicit_path:
        p = Path(explicit_path)
        if not p.is_file():
            raise FileNotFoundError("Explicit selected2200 path does not exist: " + str(p))
        candidates = [p]
    else:
        candidates = sorted(
            p
            for p in Path(input_root).rglob("*.pt")
            if p.is_file() and p.stat().st_size == SOURCE_BYTES
        )
    for p in candidates:
        print("Checking selected2200:", p, flush=True)
        if file_sha256(p) == expected_sha256:
            return p
    raise ValueError("Add Input selected2200 model weights; expected SHA256: " + expected_sha256)


def wheel_install_command(inputs, target):
    return [
        sys.executable,
        "-u",
        "-m",
        "pip",
        "install",
        "--no-index",
        "--no-deps",
        "--target",
        Path(target),
        Path(inputs) / "grast-0.1.0-py3-none-any.whl",
    ]


if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("This package requires Python 3.11–3.13.")
missing = [m for m in ("torch", "numpy", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    raise RuntimeError("Runtime is missing required preinstalled packages: " + ", ".join(missing))
INPUTS = prepare_kit(INPUT_ROOT, SCRATCH_ROOT, KIT_PATH)
INIT_WEIGHTS = resolve_source(INPUT_ROOT, SOURCE_SHA256, SOURCE_CHECKPOINT)
run_logged(wheel_install_command(INPUTS, SITE))
CHILD_ENV["PYTHONPATH"] = str(SITE) + os.pathsep + CHILD_ENV.get("PYTHONPATH", "")
run_logged(
    [
        sys.executable,
        "-u",
        "-c",
        "import sys; from pathlib import Path; import llm_scratch; "
        "assert Path(llm_scratch.__file__).resolve().is_relative_to(Path(sys.argv[2]).resolve()); "
        "from llm_scratch.easy_recovery.bundle import verify; "
        "print('Verified kit:', verify(sys.argv[1]), flush=True)",
        INPUTS,
        SITE,
    ]
)
(WORK_ROOT / "kaggle_inputs.json").write_text(
    json.dumps(
        {
            "kit_sha256": KIT_SHA256,
            "manifest_sha256": MANIFEST_SHA256,
            "source_sha256": SOURCE_SHA256,
            "source_commit": SOURCE_COMMIT,
            "kit_manifest": json.loads(
                (INPUTS / "bundle_manifest.json").read_text(encoding="utf-8")
            ),
        },
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

In [ ]:
def select_precision(capability_major, supported, saved=""):
    if saved:
        if saved not in ("float32", "float16", "bfloat16"):
            raise ValueError("Unknown saved precision")
        if saved == "bfloat16" and not (capability_major >= 8 and supported):
            raise RuntimeError("Checkpoint requires native BF16 for exact resume.")
        return saved
    return "bfloat16" if capability_major >= 8 and supported else "float16"


def resume_settings(path, inputs):
    import torch

    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError("Attach saved Notebook Output and set RESUME to latest.pt.")
    payload = torch.load(path, map_location="cpu", weights_only=True, mmap=True)
    state = payload.get("state", {})
    contract = state.get("data_state", {}).get("extra_state", {}).get("contract", {})
    manifest = json.loads((Path(inputs) / "data/manifest.json").read_text(encoding="utf-8"))
    if (
        payload.get("format_version") != 1
        or contract.get("kind") != "easy-recovery-v1"
        or contract.get("source_sha256") != SOURCE_SHA256
        or contract.get("dataset_fingerprint") != manifest["fingerprint"]
        or state.get("training_config", {}).get("max_steps") != TOTAL_UPDATES
        or type(state.get("step")) is not int
        or not 0 <= state["step"] <= TOTAL_UPDATES
    ):
        raise ValueError(
            "RESUME needs full easy-recovery latest.pt, not selected/old curriculum weights."
        )
    print("Saved runtime:", json.dumps(state["runtime"]), flush=True)
    return state["training_config"]["precision"], state["step"]


def gpu_preflight(output_root, saved=""):
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("Select a GPU accelerator; no CPU fallback.")
    major, minor = torch.cuda.get_device_capability(0)
    precision = select_precision(major, torch.cuda.is_bf16_supported(), saved)
    x = torch.randn((64, 64), device="cuda:0", requires_grad=True)
    with torch.autocast("cuda", dtype=getattr(torch, precision), enabled=precision != "float32"):
        loss = (x @ x).float().square().mean()
    loss.backward()
    torch.cuda.synchronize(0)
    if not torch.isfinite(loss) or not torch.isfinite(x.grad).all():
        raise FloatingPointError("GPU forward/backward probe failed")
    report = {
        "device": "cuda:0",
        "visible_gpus": torch.cuda.device_count(),
        "gpu": torch.cuda.get_device_name(0),
        "capability": [major, minor],
        "python": sys.version,
        "pytorch": str(torch.__version__),
        "cuda": torch.version.cuda,
        "precision": precision,
        "forward_backward_probe": "passed",
        "distributed": False,
    }
    (Path(output_root) / "kaggle_environment.json").write_text(
        json.dumps(report, indent=2) + "\n", encoding="utf-8"
    )
    print(json.dumps(report), flush=True)
    return precision


if SMOKE_ONLY and RESUME:
    raise ValueError("Set SMOKE_ONLY=False to resume a full run; smoke starts fresh.")
SAVED_PRECISION = ""
if RESUME:
    SAVED_PRECISION, RESUME_STEP = resume_settings(RESUME, INPUTS)
PRECISION = gpu_preflight(WORK_ROOT, SAVED_PRECISION)

## Training và checkpoint
Chạy mới: weights từ selected2200, optimizer/scheduler/scaler/RNG/cursor mới; giữ nguyên recipe 1.000 updates. Chọn best bằng macro verified EM rồi macro final EM trên selection16, earliest tie; step0 có thể được giữ nếu không cải thiện.

Resume: giữ cùng kit, PyTorch/CUDA, GPU, số GPU nhìn thấy và precision; đặt `SMOKE_ONLY=False`, `RESUME` thành đường dẫn full `latest.pt` trong Output đã attach. Không dùng `selected_model.pt` để resume optimizer. Runtime khác giữa Colab/Kaggle có thể không đáp ứng exact resume; runner sẽ từ chối thay vì tiếp tục âm thầm.

`/kaggle/input` chỉ đọc. Wheel và dữ liệu giải nén ở thư mục tạm, checkpoints/logs ở `/kaggle/working`. Khi lỗi hoặc interrupt, cell cố tạo summary từ đúng run đã bắt đầu; checkpoint dùng để resume là cửa sổ đã lưu gần nhất.

In [ ]:
def training_command(
    inputs, output_root, precision, smoke_only, source_checkpoint, *, resume="", device="cuda"
):
    if precision not in ("float32", "float16", "bfloat16"):
        raise ValueError("Unknown precision")
    if smoke_only and resume:
        raise ValueError("Smoke starts fresh; use SMOKE_ONLY=False for resume.")
    if not Path(source_checkpoint).is_file() or (resume and not Path(resume).is_file()):
        raise FileNotFoundError("Source weights or full resume checkpoint not found.")
    inputs = Path(inputs)
    command = [
        sys.executable,
        "-u",
        "-m",
        "llm_scratch.easy_recovery.runner",
        "--config",
        inputs / "easy_recovery.toml",
        "--prepared-root",
        inputs / "data",
        "--tokenizer",
        inputs / "tokenizer",
        "--source-checkpoint",
        Path(source_checkpoint),
        "--device",
        device,
        "--precision",
        precision,
        "--output-root",
        Path(output_root),
    ]
    if smoke_only:
        command += ["--stop-after", "2"]
    if resume:
        command += ["--resume", Path(resume)]
    return command


def export_output(run_directory, work_root):
    run, work = Path(run_directory), Path(work_root)
    if not run.resolve().is_relative_to(work.resolve()) or not run.is_dir():
        raise ValueError("Run must be the exact runner-created directory under working output.")
    summary = work / (run.name + "-summary.zip")
    with zipfile.ZipFile(summary, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(run.rglob("*")):
            if path.is_file() and path.suffix.lower() not in {
                ".pt",
                ".pth",
                ".ckpt",
                ".safetensors",
                ".u32",
            }:
                archive.write(path, path.relative_to(run).as_posix())
        for path in (CONSOLE_LOG, work / "kaggle_environment.json", work / "kaggle_inputs.json"):
            if path.is_file():
                archive.write(path, "kaggle/" + path.name)
    selected = run / "checkpoints/selected_model.pt"
    latest = run / "checkpoints/latest.pt"
    index = {
        "run_directory": str(run),
        "summary": str(summary),
        "summary_sha256": file_sha256(summary),
        "selected_model": str(selected) if selected.is_file() else None,
        "selected_sha256": file_sha256(selected) if selected.is_file() else None,
        "resume_checkpoint": str(latest) if latest.is_file() else None,
    }
    report_path = run / "artifacts/recovery_report.json"
    if report_path.is_file():
        index["report"] = json.loads(report_path.read_text(encoding="utf-8"))
    (work / "output_index.json").write_text(json.dumps(index, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(index, indent=2), flush=True)
    return index


LAST_RUN_DIRECTORY = None
OUTPUT_INDEX = None
command = training_command(INPUTS, OUTPUT_ROOT, PRECISION, SMOKE_ONLY, INIT_WEIGHTS, resume=RESUME)
try:
    run_logged(command)
finally:
    if LAST_RUN_DIRECTORY is not None:
        OUTPUT_INDEX = export_output(LAST_RUN_DIRECTORY, WORK_ROOT)
    else:
        print("No run started; inspect", CONSOLE_LOG, flush=True)
if LAST_RUN_DIRECTORY is None:
    raise RuntimeError("No exact run directory returned; do not select a stale run.")
report = OUTPUT_INDEX["report"]
if SMOKE_ONLY and (
    report["status"] != "controlled_stop" or report["step"] != 2 or report["gate_evaluated"]
):
    raise RuntimeError("Smoke acceptance failed")
if not SMOKE_ONLY and (report["status"] != "completed" or not report["gate_evaluated"]):
    raise RuntimeError("Full run is not complete")

## Lấy kết quả từ Output
Lưu version có Output để giữ các file trong `/kaggle/working`. Sau **full run**, tải **summary ZIP + selected_model.pt** (đường dẫn và checksum được in phía trên) để gửi lại kiểm tra trước WP09. Full `latest.pt` và milestones vẫn nằm trong Notebook Output để resume; summary ZIP không chứa weights.

Smoke chỉ chứng minh runner hoạt động. Gate48 chỉ chấm sau khi chọn checkpoint xong. Một vòng này kết thúc rồi chuyển WP09; không tự tăng budget hay mở thêm vòng theo điểm gate.

In [ ]:
from IPython.display import FileLink, Image, display

if OUTPUT_INDEX is None:
    raise RuntimeError("No output from the latest attempt; inspect console.log.")

for key in ("summary", "selected_model", "resume_checkpoint"):
    path = OUTPUT_INDEX.get(key)
    if path:
        print(key + ":", path, flush=True)
        display(FileLink(os.path.relpath(path, Path.cwd())))
curve = Path(OUTPUT_INDEX["run_directory"]) / "artifacts/loss_curves.png"
if curve.is_file():
    display(Image(filename=str(curve)))